# Collaborative spot the difference

In [34]:
from openai import OpenAI

from dotenv import load_dotenv
_ = load_dotenv()

from IPython.display import HTML, SVG
import re

### Generate SVG images 

In [5]:
LLM = OpenAI()

In [6]:
messages = [
    {"role": "user",
     "content": """Generate a scene in SVG format that has 40 components (shapes,  
                   objects, signs, etc.) that could be easily described by 
                   anyone. This is image A. 
                   Then create two additional versions of the same scene (image B and 
                   image C) where there are 9 differences across the three images.
                   Each of the images should have 3 of the differences. When
                   there is a difference in image A then that item should be the same
                   in the other two images.
                   Differences in components might be color or shape or a sign
                   with different words or a missing component.
                   """
    }
]

In [45]:
print(re.sub(r'\s+',' ',messages[0]['content']))

Generate a scene in SVG format that has 40 components (shapes, objects, signs, etc.) that could be easily described by anyone. This is image A. Then create two additional versions of the same scene (image B and image C) where there are 9 differences across the three images. Each of the images should have 3 of the differences. When there is a difference in image A then that item should be the same in the other two images. Differences in components might be color or shape or a sign with different words or a missing component. 


In [8]:
response = LLM.chat.completions.create(
    model="gpt-5",
    messages=messages,
    #response_format={"type": "json_object"}
)

In [15]:
print(response.choices[0].message.content)

Below are three SVG images of the same 40-component scene. Image A is the base. Images B and C each differ in exactly 3 components, for a total of 9 differences across the set. In each difference, two images match and one image is the odd one out, as requested.

Image A (SVG)
<svg xmlns="http://www.w3.org/2000/svg" width="1000" height="600" viewBox="0 0 1000 600" font-family="Arial, sans-serif">
  <!-- 1 Sky -->
  <g id="sky"><rect x="0" y="0" width="1000" height="600" fill="#B3E5FC"/></g>

  <!-- 2 Ground -->
  <g id="ground"><rect x="0" y="400" width="1000" height="200" fill="#8BC34A"/></g>

  <!-- 3 Sun (A is different: orange; B,C are yellow) -->
  <g id="sun"><circle cx="90" cy="90" r="40" fill="#FFA500" stroke="#F57C00" stroke-width="3"/></g>

  <!-- 4 Cloud 1 -->
  <g id="cloud1" fill="#FFFFFF" stroke="#E0E0E0" stroke-width="2">
    <circle cx="250" cy="120" r="30"/><circle cx="280" cy="110" r="25"/><circle cx="300" cy="125" r="28"/>
  </g>

  <!-- 5 Cloud 2 -->
  <g id="cloud2"

In [33]:
HTML('''
    <table>
        <thead>
            <tr><th>Image A</th><th>Image B</th><th>Image C</th></tr>
        </thead>
        <tbody>
            <tr>
                <td><img src="imageA.svg"/></td>
                <td><img src="imageB.svg"/></td>
                <td><img src="imageC.svg"/></td>
            </tr>
        </tbody>
    </table>
''')

Image A,Image B,Image C
,,


Notes on the 9 differences
- A-only differences (B and C match each other):
  - Sun color: A = orange; B,C = yellow.
  - Kite color: A = red; B,C = blue.
  - Direction sign text: A = “PARK”; B,C = “LAKE”.
- B-only differences (A and C match each other):
  - Bus color: B = green; A,C = yellow.
  - Tree 3: hidden in B; visible in A,C.
  - Hot air balloon color: B = purple; A,C = red.
- C-only differences (A and B match each other):
  - Lake color: C = deep blue; A,B = light blue.
  - Stop sign shape: C = square; A,B = octagon.
  - Bird 1 color: C = black; A,B = gray.

### Describe image prompt

1. load one of the SVG images and remove comments (`<-- ... -->`) as they contain information on items and differences

In [35]:
sceneA = re.sub('<!--[^>]+-->','', open('imageA.svg').read())


In [37]:
description_prompt = f'''
    Take a careful look at the scene in the image below. 
    Pay attention to the components of the image and their shape and color etc.
    Once you have looked at the image your are going to talk with 2 others who
    have both have a version of the image but there are NINE differences 
    between your three images and you have to figure them out together.

    Look at the image and make your first contribution to the discussion.

    IMAGE 

    { sceneA }

'''

In [39]:
response2 = LLM.chat.completions.create(
    model="gpt-5",
    messages=[{'role': 'user', 'content': description_prompt}]
    #response_format={"type": "json_object"}
)

In [41]:
print(response2.choices[0].message.content)

Here’s what I see in my version—calling out likely “difference hotspots.” Tell me which of these don’t match yours:

- Sun: orange circle, top-left, radius about 40.
- Clouds: three separate cloud clusters (left at ~250, middle ~430, right ~820).
- Rainbow: exactly 3 arcs (red, orange, yellow) over the right side.
- Bridge over the lake: 3 vertical posts under a brown deck.
- Direction sign: green rounded rectangle that says “PARK” in white.
- Stop sign: red octagon with white “STOP” above a gray pole.
- Fences: left fence has 6 posts (360–460); right fence has 4 posts (850–910).
- Vehicles: a blue car (2 wheels) and a yellow bus with 3 windows and 2 wheels.
- Kite: red diamond near the upper right with a string to a small dot.
- Balloon: red balloon centered at x=620 with a small brown basket below.
- Clock tower: shows 3:00 (minute hand up, hour hand pointing right).
- Animals: a dog on the left ground and a cat on the right ground.

If any of these differ on your images (counts, col